# CENG428 Neural Networks — Synthetic Shape Detection

**Framework:** PyTorch + torchvision  
**Base dataset:** MS COCO 2017  
**Task:** Detect or segment synthetic shapes added to natural images

In [ ]:
import hashlib
import random
from pathlib import Path
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFilter
from torchvision.datasets import CocoDetection
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Create the project directories used during setup, training, and evaluation.
for path in [
    Path("data"),
    Path("results"),
    Path("figures"),
    Path("figures/generated_samples"),
    Path("figures/prediction_visualizations"),
]:
    path.mkdir(parents=True, exist_ok=True)

print("Directory setup completed.")

---
## 2. Dataset and Split

In [ ]:
# Assignment constants
GLOBAL_SEED          = 2025
TRAIN_SIZE           = 5000   # first 5000 images from train2017
VAL_SIZE             = 1000   # first 1000 images from val2017
TEST_SIZE            = 1000   # next  1000 images from val2017 (indices 1000–1999)
POSITIVE_RATIO       = 0.70   # 70% positive, 30% negative
MAX_SHAPES_PER_IMAGE = 3      # 1 to 3 shapes per positive image


# Deterministic seed
# Do NOT use Python's built-in hash(); output varies between sessions.
def make_seed(split_name: str, image_id: int, global_seed: int = GLOBAL_SEED) -> int:
    key = f"{split_name}_{image_id}_{global_seed}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:8], 16)


# COCO base initialization
# Required directory structure:
#   data/coco/
#   ├── train2017/
#   ├── val2017/
#   └── annotations/
#       ├── instances_train2017.json
#       └── instances_val2017.json
def build_coco_bases(data_root: str | Path = "data/coco"):
    root = Path(data_root)

    train_base = CocoDetection(
        root=str(root / "train2017"),
        annFile=str(root / "annotations" / "instances_train2017.json"),
    )

    val_base = CocoDetection(
        root=str(root / "val2017"),
        annFile=str(root / "annotations" / "instances_val2017.json"),
    )

    return train_base, val_base


# Fixed split protocol
#   - Sort all image IDs in increasing order
#   - train : first 5000 from train2017
#   - val   : first 1000 from val2017
#   - test  : next  1000 from val2017  (do NOT use for model selection)
def get_split_ids(
    coco_base: CocoDetection,
    split: Literal["train", "val", "test"],
) -> list[int]:
    all_ids = sorted(coco_base.coco.getImgIds())

    if split == "train":
        return all_ids[:TRAIN_SIZE]
    elif split == "val":
        return all_ids[:VAL_SIZE]
    elif split == "test":
        return all_ids[VAL_SIZE : VAL_SIZE + TEST_SIZE]
    else:
        raise ValueError(f"Unknown split: {split!r}")

In [ ]:
train_base, val_base = build_coco_bases("data/coco")

train_ids = get_split_ids(train_base, "train")   # first 5000 from train2017
val_ids   = get_split_ids(val_base,   "val")     # first 1000 from val2017
test_ids  = get_split_ids(val_base,   "test")    # next  1000 from val2017

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

---
## 3. Synthetic Shape Generation

Synthetic samples are created by drawing random shapes directly onto COCO background images. The current generator supports rectangles, ellipses, and triangles, and varies location, size, color, opacity, and the number of shapes per image.

Positive images contain between **1 and 3** synthetic shapes, while negative images contain no target shape; the positive/negative ratio is approximately **70/30** as suggested in the assignment.

To keep the task nontrivial, the generator already includes multiple difficulty mechanisms such as random opacity, low-contrast colors sampled from local image content, Gaussian blur on inserted shapes, and additive noise on the final image.

Part of the synthetic shape placement and bounding-box design was inspired by prior experience from a Teknofest Aviation AI project. However, for this assignment, all synthetic shapes and their corresponding labels are generated fully automatically, and no manual annotation is used.

In [ ]:
class ShapeGenerator:
    def _sample_color(self, image_np: np.ndarray, x1: int, y1: int, x2: int, y2: int, rng: random.Random):
        patch = image_np[y1:y2, x1:x2]
        if patch.size == 0:
            patch = image_np

        base = patch.reshape(-1, 3).mean(axis=0)
        jitter = np.array([rng.randint(-35, 35) for _ in range(3)], dtype=np.float32)
        color = np.clip(base + jitter, 0, 255).astype(np.uint8)
        alpha = rng.randint(96, 220)
        return tuple(int(v) for v in color), alpha

    def _draw_shape(self, draw, shape_type: str, box, fill):
        x1, y1, x2, y2 = box
        if shape_type == "rectangle":
            draw.rectangle(box, fill=fill)
        elif shape_type == "ellipse":
            draw.ellipse(box, fill=fill)
        elif shape_type == "triangle":
            pts = [((x1 + x2) / 2, y1), (x1, y2), (x2, y2)]
            draw.polygon(pts, fill=fill)
        else:
            raise ValueError(f"Unsupported shape type: {shape_type}")

    def generate(self, image, n_shapes: int, rng: random.Random):
        """
        Draw n_shapes synthetic shapes onto image.

        Args:
            image    : PIL.Image (RGB)
            n_shapes : number of shapes to draw (0 for negative images)
            rng      : seeded random.Random instance

        Returns:
            augmented_image : PIL.Image
            boxes           : list[list[float]]   [[x1, y1, x2, y2], ...]
            mask            : np.ndarray [H, W]   1=shape pixel, 0=background
        """
        image = image.convert("RGB")
        image_np = np.array(image)
        h, w = image_np.shape[:2]

        canvas = image.convert("RGBA")
        binary_mask = np.zeros((h, w), dtype=np.uint8)
        boxes = []
        shape_types = ["rectangle", "ellipse", "triangle"]

        for _ in range(n_shapes):
            box_w = rng.randint(max(12, w // 12), max(16, w // 4))
            box_h = rng.randint(max(12, h // 12), max(16, h // 4))
            x1 = rng.randint(0, max(0, w - box_w - 1))
            y1 = rng.randint(0, max(0, h - box_h - 1))
            x2 = min(w - 1, x1 + box_w)
            y2 = min(h - 1, y1 + box_h)

            color_rgb, alpha = self._sample_color(image_np, x1, y1, x2, y2, rng)
            fill_rgba = (*color_rgb, alpha)
            shape_type = rng.choice(shape_types)
            box = (x1, y1, x2, y2)

            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            overlay_draw = ImageDraw.Draw(overlay)
            self._draw_shape(overlay_draw, shape_type, box, fill_rgba)

            blur_radius = rng.uniform(0.2, 1.6)
            overlay = overlay.filter(ImageFilter.GaussianBlur(radius=blur_radius))
            canvas = Image.alpha_composite(canvas, overlay)

            shape_mask = Image.new("L", (w, h), 0)
            mask_draw = ImageDraw.Draw(shape_mask)
            self._draw_shape(mask_draw, shape_type, box, 1)
            binary_mask = np.maximum(binary_mask, np.array(shape_mask, dtype=np.uint8))
            boxes.append([float(x1), float(y1), float(x2), float(y2)])

        augmented = np.array(canvas.convert("RGB"), dtype=np.float32)
        noise_std = rng.uniform(2.0, 8.0)
        np_rng = np.random.default_rng(rng.randint(0, 2**32 - 1))
        augmented = np.clip(augmented + np_rng.normal(0.0, noise_std, augmented.shape), 0, 255).astype(np.uint8)
        augmented_image = Image.fromarray(augmented)

        return augmented_image, boxes, binary_mask


In [ ]:
# Visualize at least 12 generated examples (§4).
# Must include both positive and negative samples.
raise NotImplementedError

---
## 4. Label Generation

Choose **Option A** (Object Detection) or **Option B** (Semantic Segmentation).

A classification-only model is **not sufficient** for full credit (§5).

In [ ]:
# PyTorch Dataset wrapper
#
# The wrapper must (§2):
#   1. Load a COCO image
#   2. Add one or more synthetic shapes
#   3. Generate the corresponding target label automatically
#   4. Return the modified image and generated target
#
# Target format — Option A: Object Detection (§5)
#   target = {
#       "boxes":       FloatTensor[N, 4]   # x_min, y_min, x_max, y_max
#       "labels":      LongTensor[N]        # all synthetic shapes share label 1
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: boxes = zeros((0,4)), labels = zeros((0,))
#
# Target format — Option B: Semantic Segmentation (§5)
#   target = {
#       "mask":        LongTensor or FloatTensor[H, W]   # 1=shape, 0=background
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: mask contains only zeros
class SyntheticShapeDataset(Dataset):
    def __init__(
        self,
        coco_base: CocoDetection,
        image_ids: list[int],
        split: Literal["train", "val", "test"],
        task: Literal["detection", "segmentation"],
        transform=None,
    ):
        self.coco_base = coco_base
        self.image_ids = image_ids
        self.split     = split
        self.task      = task
        self.transform = transform
        self.generator = ShapeGenerator()
        self.id_to_index = {img_id: i for i, img_id in enumerate(self.coco_base.ids)}

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]

        # Seed: random for train, deterministic for val/test (§3)
        if self.split == "train":
            rng = random.Random()
        else:
            rng = random.Random(make_seed(self.split, image_id))

        # Positive / negative decision — 70% positive, 30% negative (§4)
        is_positive = rng.random() < POSITIVE_RATIO
        n_shapes    = rng.randint(1, MAX_SHAPES_PER_IMAGE) if is_positive else 0

        coco_idx = self.id_to_index[image_id]
        image, _ = self.coco_base[coco_idx]
        augmented_image, boxes, mask = self.generator.generate(image, n_shapes, rng)

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        if len(boxes) == 0:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.long)
        else:
            labels_tensor = torch.ones((len(boxes),), dtype=torch.long)

        if self.transform is not None:
            augmented_image = self.transform(augmented_image)
        else:
            augmented_image = T.ToTensor()(augmented_image)

        if self.task == "detection":
            target = {
                "boxes": boxes_tensor,
                "labels": labels_tensor,
                "image_id": image_id,
                "is_positive": is_positive,
            }
        else:
            target = {
                "mask": torch.as_tensor(mask, dtype=torch.long),
                "image_id": image_id,
                "is_positive": is_positive,
            }

        return augmented_image, target

---
## 5. Model Architecture

Use a CNN-based architecture (§6). May train from scratch or fine-tune a pretrained model.
Clearly state which parts are pretrained and which are trained by you.

Detection options: Faster R-CNN, SSD-style, YOLO-style, custom CNN detector.  
Segmentation options: U-Net, FCN-style, encoder-decoder CNN, custom CNN.

In [ ]:
def build_model(task: str, pretrained: bool):
    """
    Build and return a CNN-based detection or segmentation model.
    Clearly state which parts are pretrained and which are trained from scratch.
    """
    raise NotImplementedError


TASK      = "detection"   # or "segmentation"
PRETRAINED = True
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(TASK, PRETRAINED).to(DEVICE)

---
## 6. Training Procedure

Required training details to report (§7):

| Detail | Value |
|--------|-------|
| Input image size | |
| Batch size | |
| Number of epochs | |
| Optimizer | |
| Learning rate | |
| Loss function | |
| Pretrained weights | |
| Hardware | |
| Approximate training time | |

In [ ]:
import torchvision.transforms as T

def get_transforms(split: str):
    """Return transforms for the given split. At minimum: Resize, ToTensor, Normalize."""
    raise NotImplementedError


train_ds = SyntheticShapeDataset(train_base, train_ids, "train", TASK, get_transforms("train"))
val_ds   = SyntheticShapeDataset(val_base,   val_ids,   "val",   TASK, get_transforms("val"))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def train_one_epoch(model, loader, optimizer, device) -> float:
    """Run one training epoch; return average loss."""
    raise NotImplementedError


def evaluate_val(model, loader, device) -> dict:
    """Evaluate on validation set; return metric dict. Do NOT use on test set for model selection."""
    raise NotImplementedError


import time, json
history = []

for epoch in range(1, 11):
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_val(model, val_loader, DEVICE)
    elapsed     = time.time() - t0

    print(f"Epoch {epoch:02d}  loss={train_loss:.4f}  {val_metrics}  ({elapsed:.1f}s)")
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

Path("results").mkdir(exist_ok=True)
Path("results/metrics.json").write_text(json.dumps(history, indent=2))

---
## 7. Baseline Method

Compare the CNN model against at least one simple baseline (§8).
Options: color thresholding, edge detection + connected components,
template matching, logistic regression on handcrafted features, or shallow CNN.

In [ ]:
class Baseline:
    """
    Simple baseline to compare against the CNN model (§8).
    Choose one of the options listed in §8.
    """

    def predict(self, image):
        raise NotImplementedError

---
## 8. Evaluation Metrics

In [ ]:
# Fixed test split
# Do NOT use for model selection (§7)
test_ds     = SyntheticShapeDataset(val_base, test_ids, "test", TASK, get_transforms("val"))
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4)


def _to_box_tensor(boxes) -> torch.Tensor:
    if isinstance(boxes, torch.Tensor):
        return boxes.detach().cpu().to(torch.float32)
    if boxes is None:
        return torch.zeros((0, 4), dtype=torch.float32)
    return torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def _box_iou_matrix(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    area1 = ((boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0))
    area2 = ((boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0))
    union = area1[:, None] + area2[None, :] - inter

    return torch.where(union > 0, inter / union, torch.zeros_like(inter))


def _match_boxes_greedy(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor, iou_threshold: float):
    ious = _box_iou_matrix(pred_boxes, gt_boxes)
    matches = []

    while ious.numel() > 0:
        max_iou = float(ious.max().item())
        if max_iou < iou_threshold:
            break

        flat_idx = int(ious.argmax().item())
        pred_idx = flat_idx // ious.shape[1]
        gt_idx = flat_idx % ious.shape[1]
        matches.append((pred_idx, gt_idx, max_iou))
        ious[pred_idx, :] = -1.0
        ious[:, gt_idx] = -1.0

    return matches


def detection_metrics(predictions: list, targets: list, iou_threshold: float = 0.5) -> dict:
    """
    Required (§9): Precision@0.5, Recall@0.5, F1@0.5, mean IoU of matched predictions.
    Match predicted boxes to ground-truth boxes via greedy IoU matching.
    """
    tp = 0
    fp = 0
    fn = 0
    matched_ious = []

    for pred, tgt in zip(predictions, targets):
        pred_boxes = _to_box_tensor(pred.get("boxes"))
        gt_boxes = _to_box_tensor(tgt.get("boxes"))

        if "scores" in pred and len(pred["scores"]) == len(pred_boxes):
            order = torch.argsort(pred["scores"].detach().cpu(), descending=True)
            pred_boxes = pred_boxes[order]

        matches = _match_boxes_greedy(pred_boxes, gt_boxes, iou_threshold)
        tp += len(matches)
        fp += len(pred_boxes) - len(matches)
        fn += len(gt_boxes) - len(matches)
        matched_ious.extend(match_iou for _, _, match_iou in matches)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    mean_iou = float(sum(matched_ious) / len(matched_ious)) if matched_ious else 0.0

    return {
        "precision@0.5": precision,
        "recall@0.5": recall,
        "f1@0.5": f1,
        "mean_iou_matched": mean_iou,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
    }


def segmentation_metrics(predictions: list, targets: list) -> dict:
    """
    Required (§9): foreground IoU, Dice coefficient, foreground precision, foreground recall.
    Pixel accuracy alone is NOT sufficient.
    """
    raise NotImplementedError


model.eval()
all_predictions, all_targets, all_images = [], [], []

with torch.no_grad():
    for images, targets in test_loader:
        pass  # collect predictions

if TASK == "detection":
    test_metrics = detection_metrics(all_predictions, all_targets)
else:
    test_metrics = segmentation_metrics(all_predictions, all_targets)

print("Test metrics:", test_metrics)

---
## 9. Experiments and Results

At least **3** meaningful experiments required (§10).  
For each: report quantitative results **and** a short interpretation (do not only list numbers).

Planned experiments for later runs:

1. **High opacity vs low opacity**  
   Compare two opacity ranges while keeping the model and data split fixed.
2. **Easy shapes vs hard shapes**  
   Compare simpler synthetic overlays against harder low-contrast / noisy / blurred overlays.
3. **Small training set vs larger training set**  
   Example: first 1000 training images vs first 5000 training images.
4. **With data augmentation vs without data augmentation**  
   Keep the same model and dataset, and only change the training transforms.


In [ ]:
def get_prediction_save_dir(experiment_group: str, variant: str) -> Path:
    """Return a structured directory for prediction visualizations."""
    save_dir = Path("figures") / "prediction_visualizations" / experiment_group / variant
    save_dir.mkdir(parents=True, exist_ok=True)
    return save_dir

In [ ]:
# Experiment 1 — high opacity vs low opacity
raise NotImplementedError

In [ ]:
# Experiment 2 — easy shapes vs hard shapes
raise NotImplementedError

In [ ]:
# Experiment 3 — small training set vs larger training set
raise NotImplementedError

In [ ]:
# Experiment 4 — with data augmentation vs without data augmentation
raise NotImplementedError

---
## 10. Prediction Visualizations

At least **12** test images required (§9).  
Must include: successful predictions, failure cases, positive images, negative images.

In [ ]:
def visualize_predictions(images, targets, predictions, n: int = 12, save_dir: str | Path = "figures/prediction_visualizations/general"):
    """
    Visualize at least 12 test images (§9).
    Include: successful predictions, failure cases, positive images, negative images.
    """
    raise NotImplementedError

# Example usage:
# save_dir = get_prediction_save_dir("opacity", "high")
# visualize_predictions(all_images, all_targets, all_predictions, n=12, save_dir=save_dir)

visualize_predictions(all_images, all_targets, all_predictions, n=12)

---
## 11. Failure Cases

Briefly discuss which types of synthetic additions are difficult for the model to detect and why.

---
## 12. Conclusion

Summarize findings, limitations, and possible improvements.